# Vertex AI エージェント エンジンへの ADK エージェントのデプロイ

In [ ]:
import importlib
import json
import os
import warnings

import vertexai
from vertexai.preview.reasoning_engines import AdkApp

# Ignore all warnings
warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Vertex AI API

In [ ]:
%%bash
echo > adk_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI

In [ ]:
APP_NAME = "weather_info_app"
USER_ID = "user_1"

In [ ]:
LOCATION = "us-central1"
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET_NAME = f"agent-deployment-{PROJECT}-bucket"
BUCKET_URI = f"gs://{BUCKET_NAME}"

In [ ]:
!gsutil ls $BUCKET_URI || gsutil mb -l $LOCATION $BUCKET_URI

### Initialization

In [ ]:
client = vertexai.Client(project=PROJECT, location=LOCATION)

In [ ]:
from adk_agents.agent_tmcimport agent

importlib.reload(agent)
adk_app = AdkApp(agent=agent.root_agent)

### Deploy your agent to Agent Engine

In [ ]:
DISPLAY_NAME = agent.root_agent.name
env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

remote_app = client.agent_engines.create(
    agent=adk_app,
    config={
        "display_name": DISPLAY_NAME,
        "description": "TMC_try",
        "requirements": [
            "google-adk==1.22.1",
            "google-genai==1.57.0",
            "google-cloud-aiplatform[agent_engines,adk]",
        ],
        "extra_packages": ["./adk_agents/"],
        "env_vars": env_vars,
        "staging_bucket": BUCKET_URI,
    },
)

print(f"Created remote agent: {remote_app.api_resource.display_name}")

#### Examining Session Properties output for create_session (remote):

In [ ]:
remote_session = await remote_app.async_create_session(user_id=USER_ID)

In [ ]:
remote_session